## One-dimensional Simulation of the Optical Lattice Using the Semi-Global Scheme


In [ ]:
#First, import libraries
from blochstate1d_NEW import GroundBlochState, OLConstants
import numpy as np
import time
from typing import cast
from quevolutio.core.domain import QuantumHilbertSpace, TimeGrid
from quevolutio.core.aliases import (  # isort: skip
    RVector,
    CTensors
)
from crab_propagation_tools_semiglobal import OpticalLatticeHamiltonian
from quevolutio.propagators.semi_global import ApproximationBasis, SemiGlobal
from quevolutio.core.tdse import TDSE 
import utils.numerical as numerical
import OL_visualisation as vis
from pathlib import Path

In [ ]:
#Set up the initial state
constants: OLConstants = OLConstants()
domain: QuantumHilbertSpace = QuantumHilbertSpace(
        num_dimensions=1,
        num_points=np.array([constants.num_pts]),
        position_bounds=np.array([[constants.lower_x_bound, constants.upper_x_bound]]),
        constants=constants,
    )
initial_state = GroundBlochState().generate_bloch_state()
state_initial: RVector = cast(
        RVector, domain.normalise_state(initial_state)
    )

In [ ]:
#Set up Hamiltonian, TDSE and time domain
hamiltonian = OpticalLatticeHamiltonian(domain) 
    
eigvals, eigvecs = np.linalg.eigh(GroundBlochState().H)
eigenvalue_min = np.min(eigvals)
eigenvalue_max = np.max(eigvals)

#Set up the TDSE
tdse: TDSE = TDSE(domain, hamiltonian)

# Set up the time domain.
time_domain: TimeGrid = TimeGrid(time_min=0.0, time_max=1.0, num_points=10001)
propagator = SemiGlobal(tdse, time_domain, order_m=10, order_k=10, tolerance=1e5, approximation=ApproximationBasis.NEWTONIAN)

print("Propagation Start")
start_time: float = time.time()
states: CTensors = propagator.propagate(
state_initial, diagnostics=True
    )
final_time: float = time.time()
print("Propagation Done")
runtime = final_time - start_time
print(f"Runtime: {runtime:.2f} seconds")
final_state = states[-1]

In [ ]:
#Plot the new states
# Set a common filename.
filename: str = "optical_lattice_1d"

    # Create directories for saving data if they do not exist.
for folder in (Path("data"), Path("figures"), Path("anims")):
    folder.mkdir(parents=True, exist_ok=True)

    # Save the propagated states.
np.save(f"data/{filename}.npy", states)

    # Plot the propagated states.
vis.plot_state_1D(
    states[0],
    domain,
    r"$\text{State} \; (T = 0.00)$",
    f"figures/{filename}_start.png",
    )
vis.plot_state_1D(
    states[-1],
    domain,
    rf"$\text{{State}} \; (T = {time_domain.time_axis[-1]:.2f})$",
    f"figures/{filename}_final.png",
    )


In [ ]:
#Determination of the error
# Calculate the norms of the states.
norms: RVector = numerical.states_norms(states, domain)

# Calculate the position expectation values of the states.
states_expectation: RVector = cast(
    RVector,
    np.trapezoid(
    ((np.abs(states) ** 2) * domain.position_axes[0]),
    dx=domain.position_deltas[0],
    axis=1,
        ),
    )

states_expectation/= norms

# Calculate the error from the exact position expectation values.
exact_expectation: RVector = 0.5 * (
    (time_domain.time_axis * np.cos(time_domain.time_axis))
    - np.sin(time_domain.time_axis)
)
errors: RVector = np.abs(exact_expectation - states_expectation)

# Print simulation information.
print(f"Runtime \t\t: {(final_time - start_time):.5f} seconds")
print(f"Max Error \t\t: {np.max(errors):.5e}")
print(f"Max Norm Deviation \t: {np.max(np.abs(norms - norms[0])):.5e}")
print(f"max expectation value \t: {np.max(states_expectation):.5e}")
print(np.max(exact_expectation))